# 04 – Profiles

Esplorazione e data cleaning del dataset `profiles.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (chiave primaria) |
| `gender` | Genere dichiarato |
| `birthday` | Data di nascita |
| `location` | Paese dell'utente |
| `joined` | Data di iscrizione a MAL |
| `watching` | Numero di anime attualmente in visione |
| `completed` | Numero di anime completati |
| `on_hold` | Numero di anime in pausa |
| `dropped` | Numero di anime abbandonati |
| `plan_to_watch` | Numero di anime in lista d'attesa |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_pr = pd.read_csv('../datasets/profiles.csv')
print(f'Shape: {df_pr.shape}')
print()
df_pr.info()
df_pr.head()

Il dataset contiene **337.155 righe** e **10 colonne**. I tipi di dati richiedono conversione: le colonne statistiche come `watching`, `completed`, etc sono `str` invece di `int64`, e `birthday` e `joined` sono stringhe invece di `datetime`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_pr)

mask_dup = df_pr.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pr[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pr.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

## 1.2 Rimozione profili vuoti

Verifichiamo la presenza di profili utente senza nessun dato statistico.

In [ ]:
stat_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']
mask_no_stats = df_pr[stat_cols].isna().all(axis=1)
print(f'Profili senza dati statistici : {mask_no_stats.sum():,}')
print(f'  di cui anche senza joined   : {(mask_no_stats & df_pr["joined"].isna()).sum():,}')
print()
with pd.option_context('display.expand_frame_repr', False, 'display.max_columns', None):
    print('Esempio profili vuoti:')
    print(df_pr[mask_no_stats][['username', 'gender', 'location', 'joined'] + stat_cols].head())
    print()
    print('Esempio profili vuoti con joined:')
    print(df_pr[mask_no_stats & df_pr['joined'].notna()][['username', 'gender', 'location', 'joined'] + stat_cols].head())

Dalla verifica risultano 1,678 profili senza dati statistici da cui 1,676 senza la data `joined`. Abbiamo deciso di rimuoverli in quanto non contribuiscono a nessun dato statistico e solo la location in se è un dato poco informativo.

In [ ]:
n_prima = len(df_pr)
df_pr = df_pr[~mask_no_stats].copy()
print(f'Righe prima della rimozione : {n_prima:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

## 2. Analisi colonna per colonna

### 2.1 `username`

È la **chiave primaria** del dataset. Deve essere non nulla e univoca. I duplicati non sono attesi.

In [ ]:
df_pr['username'] = df_pr['username'].str.strip()
analyze(df_pr['username'])

**Osservazioni:**
- Risulta un valore nullo in `username`. Rimuoviamo la riga in quanti un profilo senza identificatore non è utilizzabile.
- Nessun duplicato. È la chiave primaria del dataset. Nessuna ulteriore pulizia necessaria.

In [ ]:
df_pr.dropna(subset=['username'], inplace=True)
print(f'Righe dopo pulizia username         : {len(df_pr):,}')

### 2.2 `gender`

Colonna categorica con il genere dichiarato dall'utente.

In [ ]:
df_pr['gender'] = df_pr['gender'].str.strip()
analyze(df_pr['gender'])

- Si nota la presenza di 169.199 valori nulli (~50%) che non è un errore in quanto l'inserimento del genere su MAL è opzionale.
- Ci sono 3 valori unici `Male`, `Female`, `Non-Binary` che rispecchiano le opzioni presenti su MAL.

**Nessuna pulizia necessaria.**

### 2.3 `birthday`

Data di nascita dell'utente, memorizzata come stringa in formato libero (es. `'Jul 7, 1989'`, `'2003'`, `'Jan 1'`). Invece di convertire a `datetime64` — che causerebbe un'elevata perdita di dati per le date parziali — estraiamo solo l'anno di nascita come intero (`Int64`). Gli anni fuori dal range plausibile `1900–2010` vengono impostati a `null`. I null originali sono strutturali: campo opzionale su MAL.

In [ ]:
df_pr['birthday'] = df_pr['birthday'].str.strip()
analyze(df_pr['birthday'])

**Pulizia necessaria:**
- Estraiamo il primo numero a 4 cifre trovato nella stringa come anno di nascita (`\b\d{4}\b`): cattura `'1989'` da `'Jul 7, 1989'` e `'1989'` da `'1989'`, ma restituisce `null` per `'Jan 1'` (nessun anno)
- Anni fuori dal range `1900–2010` → `null` (non plausibili per un utente MAL)
- I null originali (~36%) sono strutturali

n_null_before   = df_pr['birthday'].isna().sum()
n_nonnull_before = df_pr['birthday'].notna().sum()

# Estrai anno a 4 cifre
df_pr['birthday'] = df_pr['birthday'].str.extract(r'(\b\d{4}\b)')[0].astype('Int64')

n_with_year = df_pr['birthday'].notna().sum()
n_no_year   = n_nonnull_before - n_with_year  # es. "Jan 1" → nessun anno

# Rimuovi anni fuori range plausibile
mask_invalid = df_pr['birthday'].notna() & ~df_pr['birthday'].between(1900, 2010)
n_invalid = mask_invalid.sum()
df_pr.loc[mask_invalid, 'birthday'] = pd.NA

n_null_after = df_pr['birthday'].isna().sum()

print(f'Valori non-null prima della conversione : {n_nonnull_before:,}')
print(f'  → anni estratti (1900–2010)           : {n_with_year - n_invalid:,}')
print(f'  → anni fuori range → null             : {n_invalid:,}')
print(f'  → senza anno (es. "Jan 1") → null     : {n_no_year:,}')
print(f'Null totali dopo la conversione         : {n_null_after:,} ({n_null_after / len(df_pr) * 100:.1f}%)')
print(f'birthday dtype                          : {df_pr["birthday"].dtype}')
print(f'Range                                   : {df_pr["birthday"].min()} → {df_pr["birthday"].max()}')
print()
df_pr[['username', 'birthday']].dropna(subset=['birthday']).head(10)